In [1]:
# Install required libraries
!pip install -q gradio
!pip install -q gradio_pdf
!pip install -q pypdf PyPDF2 pymupdf
!pip install -q sentence-transformers transformers
!pip install -q faiss-cpu
!pip install -q google-generativeai
!pip install -q numpy pandas

# Install LlamaIndex packages for enhanced document processing
!pip install -q llama-index
!pip install -q llama-index-readers-file
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-vector-stores-faiss
!pip install -q llama-index-llms-gemini

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.2/315.2 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.5/329.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2

# Core Imports and Configuration

In [27]:
import gradio as gr
import os
from gradio_pdf import PDF
import fitz  # PyMuPDF
from PyPDF2 import PdfReader
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import google.generativeai as genai
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
import json
from datetime import datetime
import hashlib

In [3]:
# LlamaIndex imports for enhanced document processing
from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.core.schema import TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator

# Importing Open Source and Embedding Models

In [4]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.3/137.3 kB 2.1 MB/s eta 0:00:00


In [ ]:
from groq import Groq

client = Groq(api_key="Insert your API key here")

# Initialize embedding models (both for compatibility)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
llama_embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Data Structure for Enhanced Document Management

In [16]:
@dataclass
class PageInfo:
    """Stores information about a single page"""
    page_num: int
    text: str
    doc_type: Optional[str] = None
    page_in_doc: int = 0

@dataclass
class LogicalDocument:
    """Represents a logical document within a PDF"""
    doc_id: str
    doc_type: str
    page_start: int
    page_end: int
    text: str
    chunks: List[Dict] = None

@dataclass
class ChunkMetadata:
    """Rich metadata for each chunk"""
    chunk_id: str
    doc_id: str
    doc_type: str
    chunk_index: int
    page_start: int
    page_end: int
    text: str
    embedding: Optional[np.ndarray] = None

# Document Intelligent Functions

###### Handling document classification and boundary detection

##### Document Classification

In [17]:
def classify_document_type(text: str, max_length: int = 1500) -> str:
    """
    Classify document type using Groq/Gemini.
    """
    text_sample = text[:max_length] if len(text) > max_length else text

    prompt = f"""
    Analyze this document and classify it into ONE of these categories:
    Resume, Contract, Mortgage Contract, Invoice, Pay Slip, Lender Fee Sheet,
    Land Deed, Bank Statement, Tax Document, Insurance, Report, Letter, Form,
    ID Document, Medical, Other

    Document sample:
    {text_sample}

    Respond with ONLY the category name.
    """

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200,
            temperature=0.0
        )
        doc_type = response.choices[0].message.content.strip()

        valid_types = [
            'Resume', 'Contract', 'Mortgage Contract', 'Invoice', 'Pay Slip',
            'Lender Fee Sheet', 'Land Deed', 'Bank Statement', 'Tax Document',
            'Insurance', 'Report', 'Letter', 'Form', 'ID Document', 'Medical', 'Other'
        ]
        # Match ignoring case
        for vt in valid_types:
            if doc_type.lower() == vt.lower():
                return vt
        return "Other"

    except Exception as e:
        print(f"Classification error: {e}")
        return "Other"


##### Boundary Detection

In [18]:
def detect_document_boundary(prev_text: str, curr_text: str,
                             current_doc_type: str = None) -> bool:
    """
    Detect if two consecutive pages belong to the same document.
    Returns True if same document.
    """
    if not prev_text or not curr_text:
        return False

    prev_sample = prev_text[-500:] if len(prev_text) > 500 else prev_text
    curr_sample = curr_text[:500] if len(curr_text) > 500 else curr_text

    prompt = f"""
    Determine if these two pages are from the SAME document.
    Current document type: {current_doc_type or 'Unknown'}

    End of Previous Page:
    ...{prev_sample}

    Start of Current Page:
    {curr_sample}...

    Answer ONLY 'Yes' if same document, 'No' if different document.
    """

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200,
            temperature=0.0
        )
        answer = response.choices[0].message.content.strip().lower()
        return answer.startswith("yes")
    except Exception as e:
        print(f"Boundary detection error: {e}")
        return True


# Advanced PDF Processing Pipeline

In [19]:
def extract_and_analyze_pdf(pdf_file) -> Tuple[List[PageInfo], List[LogicalDocument]]:
    """
    Extract text from PDF and perform intelligent document analysis.
    Returns both page-level info and logical document groupings.
    Supports various file types including scanned PDFs with OCR.
    """
    print("Starting PDF extraction and analysis...")

    # Extract text from each page
    if isinstance(pdf_file, dict) and "content" in pdf_file:
        doc = fitz.open(stream=pdf_file["content"], filetype="pdf")
    elif hasattr(pdf_file, "read"):
        doc = fitz.open(stream=pdf_file.read(), filetype="pdf")
    else:
        doc = fitz.open(pdf_file)

    pages_info = []
    for i, page in enumerate(doc):
        text = page.get_text()

        # If no text found, try OCR (for scanned documents)
        if not text.strip():
            print(f"  Page {i}: No text found, attempting OCR...")
            try:
                # Convert page to image and perform OCR
                pix = page.get_pixmap()
                img_data = pix.tobytes("png")
                from PIL import Image
                import pytesseract
                import io

                img = Image.open(io.BytesIO(img_data))
                text = pytesseract.image_to_string(img)
                print(f"  Page {i}: OCR extracted {len(text)} characters")
            except Exception as e:
                print(f"  Page {i}: OCR failed - {e}")
                text = ""

        pages_info.append(PageInfo(page_num=i, text=text))

    doc.close()

    if not pages_info:
        raise ValueError("No text could be extracted from PDF")

    print(f"Extracted {len(pages_info)} pages")

    # Perform document classification and boundary detection
    print("Analyzing document structure...")
    logical_docs = []
    current_doc_type = None
    current_doc_pages = []
    doc_counter = 0

    for i, page_info in enumerate(pages_info):
        if i == 0:
            # First page - classify document type
            current_doc_type = classify_document_type(page_info.text)
            page_info.doc_type = current_doc_type
            page_info.page_in_doc = 0
            current_doc_pages = [page_info]
            print(f"  Page {i}: New document detected - {current_doc_type}")
        else:
            # Check if this page continues the previous document
            prev_text = pages_info[i-1].text
            is_same = detect_document_boundary(prev_text, page_info.text, current_doc_type)

            if is_same:
                # Continue current document
                page_info.doc_type = current_doc_type
                page_info.page_in_doc = len(current_doc_pages)
                current_doc_pages.append(page_info)
            else:
                # New document detected - save previous and start new
                logical_doc = LogicalDocument(
                    doc_id=f"doc_{doc_counter}",
                    doc_type=current_doc_type,
                    page_start=current_doc_pages[0].page_num,
                    page_end=current_doc_pages[-1].page_num,
                    text="\n\n".join([p.text for p in current_doc_pages])
                )
                logical_docs.append(logical_doc)
                doc_counter += 1

                # Start new document
                current_doc_type = classify_document_type(page_info.text)
                page_info.doc_type = current_doc_type
                page_info.page_in_doc = 0
                current_doc_pages = [page_info]
                print(f"  Page {i}: New document detected - {current_doc_type}")

    # The last document
    if current_doc_pages:
        logical_doc = LogicalDocument(
            doc_id=f"doc_{doc_counter}",
            doc_type=current_doc_type,
            page_start=current_doc_pages[0].page_num,
            page_end=current_doc_pages[-1].page_num,
            text="\n\n".join([p.text for p in current_doc_pages])
        )
        logical_docs.append(logical_doc)

    print(f"Identified {len(logical_docs)} logical documents")
    for ld in logical_docs:
        print(f"   - {ld.doc_type}: Pages {ld.page_start}-{ld.page_end}")

    return pages_info, logical_docs

## Intelligent Chunking with MetaData

In [20]:
def chunk_document_with_metadata(logical_doc: LogicalDocument,
                                chunk_size: int = 500,
                                overlap: int = 100) -> List[ChunkMetadata]:
    """
    Chunk a logical document while preserving rich metadata.
    Uses sliding window with overlap for better context.
    """
    chunks_metadata = []
    words = logical_doc.text.split()

    if len(words) <= chunk_size:
        # Document is small enough to be a single chunk
        chunk_meta = ChunkMetadata(
            chunk_id=f"{logical_doc.doc_id}_chunk_0",
            doc_id=logical_doc.doc_id,
            doc_type=logical_doc.doc_type,
            chunk_index=0,
            page_start=logical_doc.page_start,
            page_end=logical_doc.page_end,
            text=logical_doc.text
        )
        chunks_metadata.append(chunk_meta)
    else:
        # Create overlapping chunks
        stride = chunk_size - overlap
        for i, start_idx in enumerate(range(0, len(words), stride)):
            end_idx = min(start_idx + chunk_size, len(words))
            chunk_text = ' '.join(words[start_idx:end_idx])

            # Calculate which pages this chunk spans
            # (simplified - in production, track more precisely)
            chunk_position = start_idx / len(words)
            page_range = logical_doc.page_end - logical_doc.page_start
            relative_page = int(chunk_position * page_range)
            chunk_page_start = logical_doc.page_start + relative_page
            chunk_page_end = min(chunk_page_start + 1, logical_doc.page_end)

            chunk_meta = ChunkMetadata(
                chunk_id=f"{logical_doc.doc_id}_chunk_{i}",
                doc_id=logical_doc.doc_id,
                doc_type=logical_doc.doc_type,
                chunk_index=i,
                page_start=chunk_page_start,
                page_end=chunk_page_end,
                text=chunk_text
            )
            chunks_metadata.append(chunk_meta)

            if end_idx >= len(words):
                break

    return chunks_metadata

def chunk_with_llama_index(logical_doc: LogicalDocument,
                           chunk_size: int = 500,
                           chunk_overlap: int = 100) -> List[Document]:

    # Create LlamaIndex document with metadata
    doc = Document(
        text=logical_doc.text,
        metadata={
            "doc_id": logical_doc.doc_id,
            "doc_type": logical_doc.doc_type,
            "page_start": logical_doc.page_start,
            "page_end": logical_doc.page_end,
            "source": f"{logical_doc.doc_type}_document"
        }
    )

    # Use LlamaIndex's sentence splitter for better chunking
    splitter = SentenceSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        paragraph_separator="\n\n",
        separator=" ",
    )

    # Create nodes (chunks) from document
    nodes = splitter.get_nodes_from_documents([doc])

    # Convert to our ChunkMetadata format for consistency
    chunks_metadata = []
    for i, node in enumerate(nodes):
        chunk_meta = ChunkMetadata(
            chunk_id=f"{logical_doc.doc_id}_chunk_{i}",
            doc_id=logical_doc.doc_id,
            doc_type=logical_doc.doc_type,
            chunk_index=i,
            page_start=node.metadata.get("page_start", logical_doc.page_start),
            page_end=node.metadata.get("page_end", logical_doc.page_end),
            text=node.text
        )
        chunks_metadata.append(chunk_meta)

    return chunks_metadata

def process_all_documents(logical_docs: List[LogicalDocument],
                         use_llama_index: bool = False) -> List[ChunkMetadata]:
    """
    Process all logical documents into chunks with metadata.
    Can use either custom or LlamaIndex chunking.
    """
    all_chunks = []

    for logical_doc in logical_docs:
        if use_llama_index:
            chunks = chunk_with_llama_index(logical_doc)
        else:
            chunks = chunk_document_with_metadata(logical_doc)

        logical_doc.chunks = chunks  # Store reference
        all_chunks.extend(chunks)
        print(f"📄 {logical_doc.doc_type}: Created {len(chunks)} chunks")

    return all_chunks

# Query Routing and Intelligent Retrieval for better Accuracy

In [21]:
def predict_query_document_type(query: str) -> Tuple[str, float]:
    prompt = f"""
    Analyze this query and predict which document type would most likely contain the answer.

    Query: "{query}"

    Choose the MOST LIKELY type from:
    Resume, Contract, Mortgage Contract, Invoice, Pay Slip, Lender Fee Sheet,
    Land Deed, Bank Statement, Tax Document, Insurance, Report, Letter,
    Form, ID Document, Medical, Other

    Respond in JSON format:
    {{"type": "DocumentType", "confidence": 0.85}}
    Confidence between 0.0 and 1.0
    """
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200,
            temperature=0.0
        )
        result_json = response.choices[0].message.content.strip()
        result = json.loads(result_json)
        return result.get("type", "Other"), result.get("confidence", 0.5)
    except Exception as e:
        print(f"Query routing error: {e}")
        return "Other", 0.0


class IntelligentRetriever:
    """
    Advanced retrieval system with metadata filtering and query routing.
    """
    def __init__(self):
        self.index = None
        self.chunks_metadata = []
        self.doc_type_indices = {}

    def build_indices(self, chunks_metadata: List[ChunkMetadata]):
        """
        Build FAISS indices with document type segregation.
        """
        print("Building vector indices...")
        self.chunks_metadata = chunks_metadata

        # Create embeddings for all chunks
        texts = [chunk.text for chunk in chunks_metadata]
        embeddings = embed_model.encode(texts, show_progress_bar=True)

        # Store embeddings in metadata
        for i, chunk in enumerate(chunks_metadata):
            chunk.embedding = embeddings[i]

        # Build main index
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(embeddings)

        # Build separate indices for each document type
        doc_types = set(chunk.doc_type for chunk in chunks_metadata)
        for doc_type in doc_types:
            type_indices = [i for i, chunk in enumerate(chunks_metadata)
                          if chunk.doc_type == doc_type]
            if type_indices:
                type_embeddings = embeddings[type_indices]
                type_index = faiss.IndexFlatL2(dim)
                type_index.add(type_embeddings)
                self.doc_type_indices[doc_type] = {
                    'index': type_index,
                    'mapping': type_indices  # Maps back to original chunks
                }

        print(f"Indexed {len(chunks_metadata)} chunks across {len(doc_types)} document types")

    def retrieve(self, query: str, k: int = 4,
                filter_doc_type: Optional[str] = None,
                auto_route: bool = True) -> List[Tuple[ChunkMetadata, float]]:
        """
        Retrieve relevant chunks with optional filtering and routing.
        Returns chunks with relevance scores.
        """
        query_embedding = embed_model.encode([query])

        # Determine which index to search
        if filter_doc_type and filter_doc_type in self.doc_type_indices:
            # Use filtered index
            type_data = self.doc_type_indices[filter_doc_type]
            D, I = type_data['index'].search(query_embedding, k)
            # Map back to original chunks
            chunk_indices = [type_data['mapping'][i] for i in I[0]]
            distances = D[0]
        elif auto_route:
            # Predict best document type
            predicted_type, confidence = predict_query_document_type(query)
            print(f"Query routed to: {predicted_type} (confidence: {confidence:.2f})")

            if confidence > 0.7 and predicted_type in self.doc_type_indices:
                # High confidence - use specific index
                type_data = self.doc_type_indices[predicted_type]
                D, I = type_data['index'].search(query_embedding, k)
                chunk_indices = [type_data['mapping'][i] for i in I[0]]
                distances = D[0]
            else:
                # Low confidence - search all
                D, I = self.index.search(query_embedding, k)
                chunk_indices = I[0]
                distances = D[0]
        else:
            # Search all chunks
            D, I = self.index.search(query_embedding, k)
            chunk_indices = I[0]
            distances = D[0]

        # Convert distances to similarity scores (inverse)
        max_dist = max(distances) if len(distances) > 0 else 1.0
        scores = [(max_dist - d) / max_dist for d in distances]

        results = [(self.chunks_metadata[i], scores[idx])
                  for idx, i in enumerate(chunk_indices)]

        return results

# Enhanced Answer generation with Source Attribution

In [22]:
def generate_answer_with_sources(query: str,
                                 retrieved_chunks: List[Tuple[ChunkMetadata, float]]) -> Dict:
    if not retrieved_chunks:
        return {'answer': "No relevant information found.", 'sources': [], 'confidence': 0.0}

    context_parts = []
    sources = []
    for chunk_meta, score in retrieved_chunks:
        context_parts.append(f"[From {chunk_meta.doc_type}, Pages {chunk_meta.page_start}-{chunk_meta.page_end}]")
        context_parts.append(chunk_meta.text)
        context_parts.append("")
        sources.append({
            'doc_type': chunk_meta.doc_type,
            'pages': f"{chunk_meta.page_start}-{chunk_meta.page_end}",
            'relevance': f"{score:.2%}",
            'preview': chunk_meta.text[:100] + "..."
        })

    context = "\n".join(context_parts)
    prompt = f"""
    You are a helpful AI assistant. Use the context to answer the question.

    Context:
    {context}

    Question: {query}

    Answer concisely and cite document type/pages.
    """

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200,
            temperature=0.2
        )
        answer = response.choices[0].message.content.strip()
        avg_score = sum(s for _, s in retrieved_chunks) / len(retrieved_chunks)
        return {'answer': answer, 'sources': sources, 'confidence': avg_score, 'chunks_used': len(retrieved_chunks)}
    except Exception as e:
        print(f"Answer generation error: {e}")
        return {'answer': f"Error: {str(e)}", 'sources': sources, 'confidence': 0.0}


# Enhanced Document Store

In [23]:
class EnhancedDocumentStore:
    """
    Manages the complete document processing and retrieval pipeline.
    """

    def __init__(self):
        self.pages_info = []
        self.logical_docs = []
        self.chunks_metadata = []
        self.retriever = IntelligentRetriever()
        self.is_ready = False
        self.processing_stats = {}
        self.filename = None

    def process_pdf(self, pdf_file, filename: str = "document.pdf"):
        """
        Complete PDF processing pipeline.
        """
        self.filename = filename
        self.is_ready = False
        start_time = datetime.now()

        try:
            # Extract and analyze PDF
            self.pages_info, self.logical_docs = extract_and_analyze_pdf(pdf_file)

            # Chunk documents with metadata
            self.chunks_metadata = process_all_documents(self.logical_docs)

            # Build retrieval indices
            self.retriever.build_indices(self.chunks_metadata)

            # Calculate processing statistics
            process_time = (datetime.now() - start_time).total_seconds()
            self.processing_stats = {
                'filename': filename,
                'total_pages': len(self.pages_info),
                'documents_found': len(self.logical_docs),
                'total_chunks': len(self.chunks_metadata),
                'document_types': list(set(doc.doc_type for doc in self.logical_docs)),
                'processing_time': f"{process_time:.1f}s"
            }

            self.is_ready = True
            return True, self.processing_stats

        except Exception as e:
            return False, {'error': str(e)}

    def query(self, question: str, filter_type: Optional[str] = None,
             auto_route: bool = True, k: int = 4) -> Dict:
        """
        Query the document store.
        """
        if not self.is_ready:
            return {
                'answer': "Please upload and process a PDF first.",
                'sources': [],
                'confidence': 0.0
            }

        # Retrieve relevant chunks
        retrieved = self.retriever.retrieve(
            question, k=k,
            filter_doc_type=filter_type,
            auto_route=auto_route
        )

        # Generate answer with sources
        result = generate_answer_with_sources(question, retrieved)
        result['filter_used'] = filter_type or ('auto' if auto_route else 'none')

        return result

    def get_document_structure(self) -> List[Dict]:
        """
        Get the document structure for UI display.
        """
        if not self.logical_docs:
            return []

        structure = []
        for doc in self.logical_docs:
            structure.append({
                'id': doc.doc_id,
                'type': doc.doc_type,
                'pages': f"{doc.page_start + 1}-{doc.page_end + 1}",
                'chunks': len(doc.chunks) if doc.chunks else 0,
                'preview': doc.text[:200] + "..." if len(doc.text) > 200 else doc.text
            })

        return structure

## Gradio Interface UI

In [62]:
# Global store instance
doc_store = EnhancedDocumentStore()

import tempfile

def process_pdf_handler(pdf_file):
    """Handle PDF upload and processing."""
    if pdf_file is None:
        return "⚠️ Please upload a PDF file", None, gr.update(choices=["All"]), None

    # Save bytes to temporary file if needed
    if isinstance(pdf_file, bytes):
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
        tmp.write(pdf_file)
        tmp.close()
        file_path = tmp.name
        filename = "document.pdf"
    else:
        file_path = pdf_file
        filename = getattr(pdf_file, "name", "document.pdf")

    success, stats = doc_store.process_pdf(file_path, filename=filename)

    if success:
        status_msg = f"""
✅ **Successfully Processed:**
- 📄 File: {stats['filename']}
- 📑 Pages: {stats['total_pages']}
- 📚 Documents Found: {stats['documents_found']}
- 🧩 Chunks Created: {stats['total_chunks']}
- 🏷️ Types: {', '.join(stats['document_types'])}
- ⏱️ Time: {stats['processing_time']}
"""
        structure = doc_store.get_document_structure()
        structure_display = "\n".join([
            f"• **{doc['type']}** (Pages {doc['pages']}): {doc['chunks']} chunks"
            for doc in structure
        ])
        doc_types = ["All"] + stats['document_types']

        return status_msg, structure_display, gr.update(choices=doc_types, value="All"), file_path
    else:
        return f"❌ Error: {stats.get('error', 'Unknown error')}", None, gr.update(choices=["All"]), None


import traceback

def chat_handler(message, history, doc_filter, auto_route, num_chunks):
    """Safe chat handler with rate-limit handling for Gradio Chatbot."""
    if history is None:
        history = []

    history.append({"role": "user", "content": str(message)})

    if not doc_store.is_ready:
        history.append({"role": "assistant", "content": "📚 Please upload and process a PDF document first."})
        return history

    filter_type = None if doc_filter == "All" else doc_filter

    try:
        result = doc_store.query(
            message,
            filter_type=filter_type,
            auto_route=auto_route and filter_type is None,
            k=num_chunks
        )

        # Ensure result is dict
        if not isinstance(result, dict):
            raise ValueError(f"doc_store.query returned invalid type: {type(result)}")

        answer = str(result.get("answer", "No answer found."))

        # Format sources
        sources = result.get("sources", [])
        if sources:
            sources_text = "\n📍 Sources:\n" + "\n".join(
                f"• {src.get('doc_type','N/A')} (Pages {src.get('pages','?')}) - Relevance: {src.get('relevance',0)}"
                for src in sources
            )
            answer += sources_text

        confidence = result.get("confidence", 0.0)
        filter_used = result.get("filter_used", "N/A")
        answer += f"\n\n*Confidence: {confidence:.1%} | Filter: {filter_used}*"

        history.append({"role": "assistant", "content": answer})
        return history

    except Exception as e:
        # Detect rate-limit errors specifically
        tb = traceback.format_exc()
        if "rate_limit_exceeded" in str(e):
            msg = ("⚠️ Rate limit reached for your API key.\n"
                   "Please wait a few minutes or use a new Groq API key.")
        else:
            msg = f"⚠️ Exception:\n{tb}"
        history.append({"role": "assistant", "content": msg})
        return history


def create_interface():
    """Create the enhanced Gradio interface with unified single-tab layout."""

    with gr.Blocks(title="Enhanced Document Q&A") as demo:
      try:
        demo.set_theme(gr.themes.Soft())
      except:
        pass
        gr.Markdown("""
        # 🚀 Enhanced Document Q&A System
        ### Intelligent Multi-Document Analysis with Advanced RAG Pipeline
        """)

        with gr.Row():
            # Left side - PDF upload and preview
            with gr.Column(scale=2):
                pdf_input = gr.File(
                    label="📄 PDF Document Viewer",
                    file_types=[".pdf"],
                    type="binary"
                )

                pdf_preview = gr.File(
                    label="📄 PDF Preview / Download",
                    interactive=False
                )

                pdf_iframe = gr.HTML(label="🔍 PDF Viewer (Embedded)")

                with gr.Row():
                    process_btn = gr.Button("🔄 Process Document", variant="primary", size="lg", scale=2)
                    clear_all_btn = gr.Button("🗑️ Clear All", variant="secondary", size="lg", scale=1)

            # Middle - Document info + settings
            with gr.Column(scale=1):
                gr.Markdown("### 📊 Document Info")
                status_output = gr.Markdown("⏳ Waiting for PDF upload...")
                structure_output = gr.Markdown("")

                gr.Markdown("### ⚙️ Settings")
                doc_filter = gr.Dropdown(choices=["All"], value="All", label="🏷️ Document Type Filter")
                auto_route = gr.Checkbox(value=True, label="🎯 Auto-Route Queries")
                num_chunks = gr.Slider(minimum=1, maximum=10, value=4, step=1, label="📊 Chunks to Retrieve")

            # Right side - Chat
            with gr.Column(scale=2):
                gr.Markdown("### 💬 Ask Questions")
                chatbot = gr.Chatbot(height=500)

                with gr.Row():
                    msg_input = gr.Textbox(placeholder="Ask your question...", show_label=False, scale=4)
                    send_btn = gr.Button("📤 Send", scale=1, variant="primary")

                with gr.Row():
                    clear_chat_btn = gr.Button("🗑️ Clear Chat", size="sm")
                    example_btn1 = gr.Button("📝 What's the summary?", size="sm")
                    example_btn2 = gr.Button("💰 Find amounts", size="sm")

        # Status bar
        status_bar = gr.Markdown("**Status:** Ready | **Documents:** 0 | **Chunks:** 0 | **Cache Hits:** 0/0")

        # Helper Functions
        def update_status_bar():
            if doc_store.is_ready:
                stats = doc_store.processing_stats
                cache_rate = 0
                if hasattr(doc_store.retriever, 'total_queries') and doc_store.retriever.total_queries > 0:
                    cache_rate = (doc_store.retriever.cache_hits / doc_store.retriever.total_queries) * 100
                return f"**Status:** ✅ Ready | **Documents:** {stats.get('documents_found',0)} | **Chunks:** {stats.get('total_chunks',0)} | **Cache Rate:** {cache_rate:.0f}%"
            return "**Status:** Ready | **Documents:** 0 | **Chunks:** 0 | **Cache Hits:** 0/0"

        def clear_all():
            global doc_store
            doc_store = EnhancedDocumentStore()
            return None, None, "", gr.update(choices=["All"], value="All"), [], "", update_status_bar()

        import base64

        def process_pdf_with_status(pdf_file):
            status, structure, filter_update, pdf_path = process_pdf_handler(pdf_file)

            iframe_html = ""
            if pdf_file:
              pdf_bytes = pdf_file.read() if hasattr(pdf_file, "read") else pdf_file
              pdf_data_url = "data:application/pdf;base64," + base64.b64encode(pdf_bytes).decode()
              iframe_html = f'<iframe src="{pdf_data_url}" width="100%" height="600px" style="border:none;"></iframe>'

            return status, structure, filter_update, update_status_bar(), None, gr.update(value=iframe_html)

        def chat_with_status(message, history, doc_filter_value, auto_route_value, num_chunks_value):
          if history is None:
            history = []

          new_history = chat_handler(message, history, doc_filter_value, auto_route_value, num_chunks_value)

          # Ensure every item is a dict with role/content
          for i, item in enumerate(new_history):
            if not isinstance(item, dict):
              new_history[i] = {"role": "assistant", "content": str(item)}
            elif "role" not in item or "content" not in item:
              new_history[i] = {"role": "assistant", "content": str(item)}
          return new_history, update_status_bar()

        def ask_summary(history):
          if history is None:
            history = []
            return chat_handler("Can you provide a summary of the main points in this document?",
                                history, doc_filter.value, auto_route.value, num_chunks.value)

        def ask_amounts(history):
          if history is None:
            history = []
            return chat_handler("What are all the monetary amounts or financial figures mentioned?",
                                history, doc_filter.value, auto_route.value, num_chunks.value)

        # Event Wiring
        process_btn.click(process_pdf_with_status, inputs=[pdf_input],
                          outputs=[status_output, structure_output, doc_filter, status_bar, pdf_preview,pdf_iframe])
        pdf_input.change(process_pdf_with_status, inputs=[pdf_input],
                         outputs=[status_output, structure_output, doc_filter, status_bar, pdf_preview,pdf_iframe])
        clear_all_btn.click(clear_all, outputs=[pdf_input, pdf_preview, status_output,
                                                doc_filter, chatbot, msg_input, status_bar])

        msg_input.submit(
          fn=chat_with_status,
          inputs=[msg_input, chatbot, doc_filter, auto_route, num_chunks],
          outputs=[chatbot, status_bar]
          ).then(lambda: "", outputs=[msg_input])

        send_btn.click(
          fn=chat_with_status,
          inputs=[msg_input, chatbot, doc_filter, auto_route, num_chunks],
          outputs=[chatbot, status_bar]
          ).then(lambda: "", outputs=[msg_input])

        clear_chat_btn.click(lambda: [], outputs=[chatbot])
        example_btn1.click(ask_summary, inputs=[chatbot], outputs=[chatbot]).then(update_status_bar, outputs=[status_bar])
        example_btn2.click(ask_amounts, inputs=[chatbot], outputs=[chatbot]).then(update_status_bar, outputs=[status_bar])

    return demo


In [65]:
demo = create_interface()
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c9cf342be39cc8b9bd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c9cf342be39cc8b9bd.gradio.live
